In [5]:
import os
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import cv2
import random

## Preprocess images

In [6]:
# Folder paths
up_folder = "bridgeUp"
down_folder = "bridgeDown"

# Store features and labels
X = []
y = []

# Use the same image dimensions for every image
image_size = (1080, 1080)

# Load bridge-up images (label 1)
for filename in os.listdir(up_folder):
    filepath = os.path.join(up_folder, filename)

    try:
        image = Image.open(filepath).convert("L")
        image = image.resize(image_size)
        image = np.array(image)

        features = hog(
            image,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        )

        X.append(features)
        y.append(1)

    except (OSError, ValueError):
        continue


In [7]:
# Load 500 randomly selected bridge-down images (label 0)
down_files = os.listdir(down_folder)
down_files = random.sample(down_files, 500)

for filename in down_files:
    filepath = os.path.join(down_folder, filename)

    try:
        image = Image.open(filepath).convert("L")
        image = image.resize(image_size)
        image = np.array(image)

        features = hog(
            image,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        )

        X.append(features)
        y.append(0)

    except (OSError, ValueError):
        continue

In [8]:
# Convert to NumPy arrays
X = np.array(X)
y = np.array(y)

print("Number of images:", len(y))
print("Feature matrix shape:", X.shape)
print("Label array shape:", y.shape)
print("Up images:", np.sum(y == 1))
print("Down images:", np.sum(y == 0))


Number of images: 649
Feature matrix shape: (649, 646416)
Label array shape: (649,)
Up images: 149
Down images: 500


## Split into training and testing sets

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))

Training images: 519
Testing images: 130


In [10]:
model = LinearSVC(C=1, max_iter=5000)

model.fit(X_train, y_train)

LinearSVC(C=1, max_iter=5000)

## Use testing data on model

In [11]:
y_train_pred = model.predict(X_train)

training_error = np.mean(y_train_pred != y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))

testing_error = np.mean(y_pred != y_test)
print("Testing Error: ", testing_error)
print("Training Error: ",training_error )

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Bridge Down", "Bridge Up"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9923076923076923
Testing Error:  0.007692307692307693
Training Error:  0.0

Classification Report:
              precision    recall  f1-score   support

 Bridge Down       0.99      1.00      1.00       100
   Bridge Up       1.00      0.97      0.98        30

    accuracy                           0.99       130
   macro avg       1.00      0.98      0.99       130
weighted avg       0.99      0.99      0.99       130


Confusion Matrix:
[[100   0]
 [  1  29]]


## Predict on new image

In [25]:
# Path to the new image
image_path = "testImage.jpg" #PUT IMAGE HERE

# Load and preprocess the image
image = Image.open(image_path).convert("L")
image = image.resize(image_size)
image = np.array(image)

# Extract HOG features using the same settings
features = hog(
    image,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys"
)

# Convert features into the format expected by the model
features = features.reshape(1, -1)

# Predict the bridge position
prediction = model.predict(features)[0]

# Display the result
if prediction == 1:
    print("Prediction: Bridge Up")
else:
    print("Prediction: Bridge Down")

Prediction: Bridge Up


## Predict on Video Footage

In [13]:
# Path to the video
video_path = "test5Trim.mp4"

# Open the video
cap = cv2.VideoCapture(video_path)


while cap.isOpened():

    # Read the next frame
    ret, frame = cap.read()

    if not ret:

        break

    # Resize frame to match training images
    frame = cv2.resize(frame, image_size)

    # Convert frame to grayscale
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Extract HOG features using the same settings as training
    features = hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    # Reshape features for prediction
    features = features.reshape(1, -1)

    # Predict bridge position
    prediction = model.predict(features)[0]

    # Display prediction
    if prediction == 1:
        label = "Bridge Up"
        color = (0, 255, 0)
    else:
        label = "Bridge Down"
        color = (0, 0, 255)

    # Add prediction to frame
    cv2.putText(
        frame,
        label,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        color,
        2
    )
    

    # Display video
    cv2.imshow("Bridge Position Prediction", frame)

    # Press q to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Release video
cap.release()
cv2.destroyAllWindows()

## Measure Computational Cost

In [14]:
import time

Process and Predict One Image

In [17]:
# Path to the new image
image_path = "testImage.jpg"

# Start timing
start_time = time.perf_counter()

# Load and preprocess the image
image = Image.open(image_path).convert("L")
image = image.resize(image_size)
image = np.array(image)

# Extract HOG features
features = hog(
    image,
    orientations=9,
    pixels_per_cell=(8, 8),
    cells_per_block=(2, 2),
    block_norm="L2-Hys"
)

# Convert features into the format expected by the model
features = features.reshape(1, -1)

# Measure total prediction pipeline time
preprocessing_end = time.perf_counter()

# Predict the bridge position
prediction = model.predict(features)[0]

end_time = time.perf_counter()

# Calculate computational time
preprocessing_time = preprocessing_end - start_time
prediction_time = end_time - preprocessing_end
total_time = end_time - start_time

print("Preprocessing Time:", preprocessing_time, "seconds")
print("SVM Prediction Time:", prediction_time, "seconds")
print("Total Computational Time:", total_time, "seconds")

# Display the result
if prediction == 1:
    print("Prediction: Bridge Up")
else:
    print("Prediction: Bridge Down")

Preprocessing Time: 0.3026594000402838 seconds
SVM Prediction Time: 0.0012459999416023493 seconds
Total Computational Time: 0.30390539998188615 seconds
Prediction: Bridge Up


Process and predict the image 100 times and take average

In [26]:
num_iterations = 100

preprocessing_times = []
hog_times = []
prediction_times = []
total_times = []

for i in range(num_iterations):

    total_start = time.perf_counter()

    # 1. Load image
    start_time = time.perf_counter()

    image = Image.open(image_path).convert("L")
    image = image.resize(image_size)
    image = np.array(image)

    preprocessing_end = time.perf_counter()

    # 2. Extract HOG features
    features = hog(
        image,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys"
    )

    hog_end = time.perf_counter()

    # 3. SVM prediction
    features = features.reshape(1, -1)

    prediction = model.predict(features)[0]

    prediction_end = time.perf_counter()

    # Store times
    preprocessing_times.append(preprocessing_end - start_time)
    hog_times.append(hog_end - preprocessing_end)
    prediction_times.append(prediction_end - hog_end)
    total_times.append(prediction_end - total_start)


# Calculate averages
print("Average Image Preprocessing Time:",
      np.mean(preprocessing_times), "seconds")

print("Average HOG Feature Extraction Time:",
      np.mean(hog_times), "seconds")

print("Average SVM Prediction Time:",
      np.mean(prediction_times), "seconds")

print("Average Total Prediction Pipeline Time:",
      np.mean(total_times), "seconds")

# Display prediction
if prediction == 1:
    print("Prediction: Bridge Up")
else:
    print("Prediction: Bridge Down")

Average Image Preprocessing Time: 0.01000760299502872 seconds
Average HOG Feature Extraction Time: 0.3976846250030212 seconds
Average SVM Prediction Time: 0.0012968370027374475 seconds
Average Total Prediction Pipeline Time: 0.4089892569987569 seconds
Prediction: Bridge Up
